# In this notebook we are going to see how torch does in solving a regression problem using a gaussian process

## 0. Imports and helpers

In [1]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# --- Toy data ---
def f_true(x):
    return np.sin(x)

n_train = 60
X_train = rng.uniform(-3.0, 3.0, size=(n_train, 1))
y_train = f_true(X_train) + rng.normal(0, 0.2, size=(n_train, 1))

X_test = np.linspace(-3.5, 3.5, 200)[:, None]
y_true = f_true(X_test)

results = []

# --- Helper: compute RMSE and MNLP ---
def eval_metrics(mu, var, y_true):
    mu = mu.reshape(-1, 1)
    var = var.reshape(-1, 1)
    rmse = float(np.sqrt(np.mean((mu - y_true)**2)))
    # Mean Negative Log Predictive density assuming Gaussian predictive distribution
    mnlp = float(0.5*np.mean(np.log(2*np.pi*var) + (y_true - mu)**2/var))
    return rmse, mnlp

## 1. Build using torch

In [1]:
import torch
print(torch.cuda.is_available())        # True if CUDA works
print(torch.cuda.get_device_name(0))

True
NVIDIA T400 4GB
